In [1]:
!nvidia-smi

Sat Jul 18 16:34:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   45C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
!pip install -q transformers librosa soundfile accelerate

In [3]:
from google.colab import drive;

In [4]:
drive.mount('/content/drive')


Mounted at /content/drive


In [5]:
!ls "/content/drive/MyDrive/Context AI/processed"

clean  manifest.jsonl  pipeline.log  segments  transcription.log


In [6]:
import os, re, json, time
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

BASE = "/content/drive/MyDrive/Context AI/processed"
MANIFEST_PATH = f"{BASE}/manifest.jsonl"
SEGMENTS_DIR = f"{BASE}/segments"
TRANSCRIPT_LOG = f"{BASE}/transcription.log"

HF_ASR_MODEL_ID = "openai/whisper-large-v3"
TARGET_SR = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def load_manifest():
    records = []
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def save_manifest(records):
    tmp = MANIFEST_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, MANIFEST_PATH)

records = load_manifest()

# --- RESUME LOGIC: skip anything already done by large-v3 in yesterday's run ---
pending = [r for r in records if not (r.get("transcript_model") == HF_ASR_MODEL_ID and r.get("transcript"))]
already_done = len(records) - len(pending)
total = len(pending)
print(f"Total segments={len(records)} | already done (yesterday)={already_done} | remaining to-process={total}")

if total == 0:
    print("Nothing left — everything is already transcribed with large-v3!")
else:
    print("Loading model (float32, may take a few minutes)...")
    processor = WhisperProcessor.from_pretrained(HF_ASR_MODEL_ID)
    model = WhisperForConditionalGeneration.from_pretrained(HF_ASR_MODEL_ID, torch_dtype=torch.float32)
    model = model.to(device)
    print("Model loaded on", device)

    def transcribe_one(wav_path):
        y, _ = librosa.load(wav_path, sr=TARGET_SR, mono=True)
        inputs = processor(y, sampling_rate=TARGET_SR, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            generated = model.generate(
                **inputs,
                language="urdu",
                task="transcribe",
                num_beams=3,
                max_new_tokens=225,
            )
        text = processor.batch_decode(generated, skip_special_tokens=True)[0]
        return text.strip()

    done = 0
    n_errors = 0
    start = time.time()

    with open(TRANSCRIPT_LOG, "a", encoding="utf-8") as lf:
        lf.write(f"[Colab GPU run RESUMED] {already_done} already done, {total} remaining\n")

    for r in pending:
        seg_id = r.get("segment_id")
        fname = r.get("filename")
        source_type = r.get("source_type", "")
        wav = f"{SEGMENTS_DIR}/{source_type}/{fname}"
        if not os.path.exists(wav):
            wav = f"{SEGMENTS_DIR}/{fname}"
        if not os.path.exists(wav):
            print(f"MISSING: {fname}")
            n_errors += 1
            done += 1
            continue
        try:
            text = transcribe_one(wav)
            r["transcript"] = text
            r["transcript_model"] = HF_ASR_MODEL_ID
            r["verified"] = False
        except Exception as e:
            print(f"ERROR on {seg_id}: {e}")
            n_errors += 1

        done += 1
        if done % 10 == 0 or done == total:
            save_manifest(records)
            elapsed = time.time() - start
            rate = done / elapsed if elapsed else 0
            eta = (total - done) / rate / 60 if rate else 0
            msg = f"{done}/{total} done ({100*done/total:.1f}%) | {rate:.2f} seg/s | ETA {eta:.1f} min"
            print(msg)
            with open(TRANSCRIPT_LOG, "a", encoding="utf-8") as lf:
                lf.write(msg + "\n")

    save_manifest(records)
    print(f"DONE. remaining_processed={done - n_errors} errors={n_errors} already_done_before={already_done}")

Using device: cuda
Total segments=903 | already done (yesterday)=710 | remaining to-process=193
Loading model (float32, may take a few minutes)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

180/193 done (93.3%) | 0.08 seg/s | ETA 2.6 min


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


190/193 done (98.4%) | 0.08 seg/s | ETA 0.6 min


Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=225) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


193/193 done (100.0%) | 0.08 seg/s | ETA 0.0 min
DONE. remaining_processed=193 errors=0 already_done_before=710


In [ ]:
import os, re, json, time
import torch
import librosa
from transformers import WhisperProcessor, WhisperForConditionalGeneration

BASE = "/content/drive/MyDrive/Context AI/processed"
MANIFEST_PATH = f"{BASE}/manifest.jsonl"
SEGMENTS_DIR = f"{BASE}/segments"
TRANSCRIPT_LOG = f"{BASE}/transcription.log"

HF_ASR_MODEL_ID = "openai/whisper-large-v3"
TARGET_SR = 16000

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

def load_manifest():
    records = []
    with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def save_manifest(records):
    tmp = MANIFEST_PATH + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")
    os.replace(tmp, MANIFEST_PATH)

records = load_manifest()
total = len(records)
print(f"Total segments to transcribe (ALL, fresh): {total}")

print("Loading model (float32, may take a few minutes)...")
processor = WhisperProcessor.from_pretrained(HF_ASR_MODEL_ID)
model = WhisperForConditionalGeneration.from_pretrained(HF_ASR_MODEL_ID, torch_dtype=torch.float32)
model = model.to(device)
print("Model loaded on", device)

def transcribe_one(wav_path):
    y, _ = librosa.load(wav_path, sr=TARGET_SR, mono=True)
    inputs = processor(y, sampling_rate=TARGET_SR, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        generated = model.generate(
            **inputs,
            language="urdu",
            task="transcribe",
            num_beams=3,
            max_new_tokens=225,
        )
    text = processor.batch_decode(generated, skip_special_tokens=True)[0]
    return text.strip()

done = 0
n_errors = 0
start = time.time()

with open(TRANSCRIPT_LOG, "a", encoding="utf-8") as lf:
    lf.write(f"[Colab GPU run - FULL 903, large-v3, fresh restart] 0/{total} segments\n")

for r in records:
    seg_id = r.get("segment_id")
    fname = r.get("filename")
    source_type = r.get("source_type", "")
    wav = f"{SEGMENTS_DIR}/{source_type}/{fname}"
    if not os.path.exists(wav):
        wav = f"{SEGMENTS_DIR}/{fname}"
    if not os.path.exists(wav):
        print(f"MISSING: {fname}")
        n_errors += 1
        done += 1
        continue
    try:
        text = transcribe_one(wav)
        r["transcript"] = text
        r["transcript_model"] = HF_ASR_MODEL_ID
        r["verified"] = False   # reset — sab dobara fresh check hone chahiye
    except Exception as e:
        print(f"ERROR on {seg_id}: {e}")
        n_errors += 1

    done += 1
    if done % 10 == 0 or done == total:
        save_manifest(records)
        elapsed = time.time() - start
        rate = done / elapsed if elapsed else 0
        eta = (total - done) / rate / 60 if rate else 0
        msg = f"{done}/{total} done ({100*done/total:.1f}%) | {rate:.2f} seg/s | ETA {eta:.1f} min"
        print(msg)
        with open(TRANSCRIPT_LOG, "a", encoding="utf-8") as lf:
            lf.write(msg + "\n")

save_manifest(records)
print(f"DONE. total={total} errors={n_errors}")